# Data Preparation

In [1]:
import os
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from datasets import load_dataset

PROJECT_ROOT = '/content/drive/MyDrive/twincar'
STANFORD_CACHE = f'{PROJECT_ROOT}/stanford_cars_cache'
COMPCARS_PATH = f'{PROJECT_ROOT}/compcars_filtered'
DATA_DIR = f'{PROJECT_ROOT}/data'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Load Stanford Cars & Extract Class Map

In [3]:
dataset = load_dataset('naufalso/stanford_cars', cache_dir=STANFORD_CACHE)

train_data = dataset['train']
test_data  = dataset['test']

train_records = [{'hf_idx': i, 'label': ex['label'], 'car_name': ex['car_name']}
                 for i, ex in enumerate(train_data)]
test_records  = [{'hf_idx': i, 'label': ex['label'], 'car_name': ex['car_name']}
                 for i, ex in enumerate(test_data)]

train_df = pd.DataFrame(train_records)
test_df  = pd.DataFrame(test_records)

train_df["source_split"] = "train"
test_df["source_split"] = "test"

del dataset, train_data, test_data

print("Train rows: " + str(len(train_df)))
print("Test rows: " + str(len(test_df)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/614 [00:00<?, ?B/s]

Train rows: 8103
Test rows: 8000


In [4]:
print("Unique labels in train_df:", train_df["label"].nunique())
print("Unique labels in test_df:", test_df["label"].nunique())

all_train_labels = set(train_df["label"].unique())
all_test_labels = set(test_df["label"].unique())

print("Labels only in train:", sorted(all_train_labels - all_test_labels))
print("Labels only in test:", sorted(all_test_labels - all_train_labels))
print("Total unique labels:", len(all_train_labels | all_test_labels))

Unique labels in train_df: 195
Unique labels in test_df: 195
Labels only in train: []
Labels only in test: []
Total unique labels: 195


In [5]:
# Check that labels can be safely mapped to class_id
original_labels = sorted(train_df["label"].unique())

label_to_class_id = {
    original_label: new_id
    for new_id, original_label in enumerate(original_labels)
}

train_df["class_id"] = train_df["label"].map(label_to_class_id)
test_df["class_id"] = test_df["label"].map(label_to_class_id)

print("Train class_id min:", train_df["class_id"].min())
print("Train class_id max:", train_df["class_id"].max())
print("Test class_id min:", test_df["class_id"].min())
print("Test class_id max:", test_df["class_id"].max())

print("Missing class_id in train:", train_df["class_id"].isna().sum())
print("Missing class_id in test:", test_df["class_id"].isna().sum())
print("Number of class_ids:", len(label_to_class_id))

Train class_id min: 0
Train class_id max: 194
Test class_id min: 0
Test class_id max: 194
Missing class_id in train: 0
Missing class_id in test: 0
Number of class_ids: 195


### Parse Class Names into Make/Model/Year

In [6]:
MULTI_WORD_MAKES = [
    "AM General",
    "Aston Martin",
    "Land Rover"
]

def parse_car_name(name):
    """
    Parse Stanford Cars class name into make, model, and year.

    Example:
        "Audi S4 Sedan 2012"
        -> make="Audi", model="S4 Sedan", year="2012"

        "Aston Martin V8 Vantage Convertible 2012"
        -> make="Aston Martin", model="V8 Vantage Convertible", year="2012"
    """
    name = name.strip()
    tokens = name.split()

    # Last token is usually the year
    if tokens[-1].isdigit():
        year = tokens[-1]
        name_without_year = " ".join(tokens[:-1])
    else:
        year = None
        name_without_year = name

    # Handle known multi-word makes
    for make_candidate in MULTI_WORD_MAKES:
        if name_without_year.startswith(make_candidate + " "):
            make = make_candidate
            model = name_without_year.replace(make_candidate + " ", "", 1)
            make_model = f"{make} {model}"
            return make, model, make_model, year

    # Default case
    tokens_without_year = name_without_year.split()
    make = tokens_without_year[0]
    model = " ".join(tokens_without_year[1:])
    make_model = f"{make} {model}"

    return make, model, make_model, year


# Build original label -> class name mapping
label_to_name = {}

for _, row in train_df.iterrows():
    label_to_name[row["label"]] = row["car_name"]

for _, row in test_df.iterrows():
    label_to_name[row["label"]] = row["car_name"]


# Create clean class_id mapping: 0 to num_classes - 1
original_labels = sorted(label_to_name.keys())

label_to_class_id = {
    original_label: class_id
    for class_id, original_label in enumerate(original_labels)
}


# Add class_id to train and test dataframes
train_df["class_id"] = train_df["label"].map(label_to_class_id)
test_df["class_id"] = test_df["label"].map(label_to_class_id)


# Create class dataframe
rows = []

for original_label in original_labels:
    class_id = label_to_class_id[original_label]
    class_name = label_to_name[original_label]

    make, model, make_model, year = parse_car_name(class_name)

    rows.append({
        "original_label": original_label,
        "class_id": class_id,
        "class_name": class_name,
        "make": make,
        "model": model,
        "make_model": make_model,
        "year": int(year) if year is not None else None
    })

class_df = pd.DataFrame(rows)

print("Number of classes:", class_df["class_id"].nunique())
print("Unique makes:", class_df["make"].nunique())
print("Unique make_model combinations:", class_df["make_model"].nunique())
print("Unique models:", class_df["model"].nunique())
print("Year range:", str(class_df["year"].min()) + " - " + str(class_df["year"].max()))

print("\nExample parsed classes:")
class_df.head()

Number of classes: 195
Unique makes: 48
Unique make_model combinations: 188
Unique models: 188
Year range: 1991 - 2012

Example parsed classes:


,original_label,class_id,class_name,make,model,make_model,year
0,1,0,AM General Hummer SUV 2000,AM General,Hummer SUV,AM General Hummer SUV,2000
1,2,1,Acura RL Sedan 2012,Acura,RL Sedan,Acura RL Sedan,2012
2,3,2,Acura TL Sedan 2012,Acura,TL Sedan,Acura TL Sedan,2012
3,4,3,Acura TL Type-S 2008,Acura,TL Type-S,Acura TL Type-S,2008
4,5,4,Acura TSX Sedan 2012,Acura,TSX Sedan,Acura TSX Sedan,2012


### Build Train/Val/Test Split from Stanford Cars

The StanfordCars dataset comes with a predefined `train` (8,103 images) and `test` (8,000 images) split.
We keep the `test` split as-is and carve a validation set out of the `train` split.

**Split strategy:**
- `test`: StanfordCars test split (8,000 images, untouched)
- `train`: 85% of StanfordCars train split (~6,887 images)
- `val`: 15% of StanfordCars train split (~1,216 images)

Splitting is **stratified by class label** to ensure every class is represented
proportionally in both train and val, which is important given the class imbalance
we observed in the EDA (min 24, max 68 images per class).

`random_state=42` ensures the split is reproducible.

In [7]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df['label'],
    random_state=SEED
)

train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)

print("Stanford split:")
print("  Train : " + str(len(train_split)))
print("  Val: " + str(len(val_split)))
print("  Test: " + str(len(test_df)))
print("  Total: " + str(len(train_split) + len(val_split) + len(test_df)))
print("\nClasses in train: " + str(train_split['label'].nunique()))
print("Classes in val: " + str(val_split['label'].nunique()))

Stanford split:
  Train : 6887
  Val: 1216
  Test: 8000
  Total: 16103

Classes in train: 195
Classes in val: 195


In [8]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label"],
    random_state=SEED
)

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Stanford split:")
print("  Train:", len(train_split))
print("  Val  :", len(val_split))
print("  Test :", len(test_df))
print("  Total:", len(train_split) + len(val_split) + len(test_df))

print("\nClasses per split:")
print("  Train:", train_split["label"].nunique())
print("  Val  :", val_split["label"].nunique())
print("  Test :", test_df["label"].nunique())

print("\nClass distribution check:")
print("  Min train images per class:", train_split["label"].value_counts().min())
print("  Min val images per class  :", val_split["label"].value_counts().min())
print("  Min test images per class :", test_df["label"].value_counts().min())

Stanford split:
  Train: 6887
  Val  : 1216
  Test : 8000
  Total: 16103

Classes per split:
  Train: 195
  Val  : 195
  Test : 195

Class distribution check:
  Min train images per class: 20
  Min val images per class  : 4
  Min test images per class : 24


In [11]:
print("Class ID ranges:")
print("  Train:", train_split["class_id"].min(), "-", train_split["class_id"].max())
print("  Val  :", val_split["class_id"].min(), "-", val_split["class_id"].max())
print("  Test :", test_df["class_id"].min(), "-", test_df["class_id"].max())

print("\nMissing class_id values:")
print("  Train:", train_split["class_id"].isna().sum())
print("  Val  :", val_split["class_id"].isna().sum())
print("  Test :", test_df["class_id"].isna().sum())

train_indices = set(train_split["hf_idx"])
val_indices = set(val_split["hf_idx"])

assert train_indices.isdisjoint(val_indices), "Data leakage: train and val overlap!"

combined_check = pd.concat(
    [train_split, val_split, test_df],
    ignore_index=True
).copy()

combined_check["unique_source_key"] = (
    combined_check["source_split"].astype(str)
    + "_"
    + combined_check["hf_idx"].astype(str)
)

assert combined_check["unique_source_key"].nunique() == len(combined_check), (
    "Duplicate images found across splits!"
)

print("\nSafety checks passed.")
print("No train/val overlap.")
print("No duplicate source images across train/val/test.")

Class ID ranges:
  Train: 0 - 194
  Val  : 0 - 194
  Test : 0 - 194

Missing class_id values:
  Train: 0
  Val  : 0
  Test : 0

Safety checks passed.
No train/val overlap.
No duplicate source images across train/val/test.


In [12]:
os.makedirs(DATA_DIR, exist_ok=True)

train_split = train_split.copy()
val_split = val_split.copy()
test_df = test_df.copy()

train_split["split"] = "train"
val_split["split"] = "val"
test_df["split"] = "test"

metadata = pd.concat(
    [train_split, val_split, test_df],
    ignore_index=True
)

# Add class information: class_name, make, model, make_model, year
class_info = class_df[
    ["original_label", "class_id", "class_name", "make", "model", "make_model", "year"]
].copy()

metadata = metadata.merge(
    class_info,
    left_on="label",
    right_on="original_label",
    how="left",
    suffixes=("", "_from_class_df")
)

# Check that class_id from metadata and class_df match
assert (
    metadata["class_id"] == metadata["class_id_from_class_df"]
).all(), "class_id mismatch after merge!"

# Remove duplicate helper column
metadata = metadata.drop(columns=["class_id_from_class_df"])

# Safety checks
required_columns = [
    "hf_idx",
    "label",
    "class_id",
    "car_name",
    "class_name",
    "make",
    "model",
    "make_model",
    "year",
    "source_split",
    "split"
]

missing_columns = [col for col in required_columns if col not in metadata.columns]
assert len(missing_columns) == 0, f"Missing columns: {missing_columns}"

print("Missing values:")
print(metadata[["class_id", "class_name", "make", "model", "make_model", "year"]].isna().sum())

print("\nSplit counts:")
print(metadata["split"].value_counts())

print("\nNumber of classes:")
print(metadata["class_id"].nunique())

print("\nNumber of make/model combinations:")
print(metadata["make_model"].nunique())

metadata_path = os.path.join(DATA_DIR, "stanford_metadata.csv")
metadata.to_csv(metadata_path, index=False)

print("\nSaved metadata to:", metadata_path)
print("Total rows:", len(metadata))

metadata.head()

Missing values:
class_id      0
class_name    0
make          0
model         0
make_model    0
year          0
dtype: int64

Split counts:
split
test     8000
train    6887
val      1216
Name: count, dtype: int64

Number of classes:
195

Number of make/model combinations:
188

Saved metadata to: /content/drive/MyDrive/twincar/data/stanford_metadata.csv
Total rows: 16103


,hf_idx,label,car_name,source_split,class_id,split,original_label,class_name,make,model,make_model,year
0,4369,102,Ferrari California Convertible 2012,train,101,train,102,Ferrari California Convertible 2012,Ferrari,California Convertible,Ferrari California Convertible,2012
1,1962,190,Volkswagen Golf Hatchback 2012,train,188,train,190,Volkswagen Golf Hatchback 2012,Volkswagen,Golf Hatchback,Volkswagen Golf Hatchback,2012
2,262,69,Chevrolet Silverado 2500HD Regular Cab 2012,train,68,train,69,Chevrolet Silverado 2500HD Regular Cab 2012,Chevrolet,Silverado 2500HD Regular Cab,Chevrolet Silverado 2500HD Regular Cab,2012
3,2742,104,Ferrari 458 Italia Coupe 2012,train,103,train,104,Ferrari 458 Italia Coupe 2012,Ferrari,458 Italia Coupe,Ferrari 458 Italia Coupe,2012
4,7602,79,Chrysler 300 SRT-8 2010,train,78,train,79,Chrysler 300 SRT-8 2010,Chrysler,300 SRT-8,Chrysler 300 SRT-8,2010


In [13]:
class_map_path = os.path.join(DATA_DIR, "stanford_class_map.csv")
class_df.to_csv(class_map_path, index=False)

print("Saved class map to:", class_map_path)

Saved class map to: /content/drive/MyDrive/twincar/data/stanford_class_map.csv


In [15]:
import json

label_map = {
    int(row["class_id"]): {
        "class_name": row["class_name"],
        "make": row["make"],
        "model": row["model"],
        "make_model": row["make_model"],
        "year": int(row["year"]) if pd.notna(row["year"]) else None
    }
    for _, row in class_df.iterrows()
}

label_map_path = os.path.join(DATA_DIR, "label_map.json")

with open(label_map_path, "w") as f:
    json.dump(label_map, f, indent=4)

print("Saved label map to:", label_map_path)

# Quick preview
list(label_map.items())[:3]

Saved label map to: /content/drive/MyDrive/twincar/data/label_map.json


[(0,
  {'class_name': 'AM General Hummer SUV 2000',
   'make': 'AM General',
   'model': 'Hummer SUV',
   'make_model': 'AM General Hummer SUV',
   'year': 2000}),
 (1,
  {'class_name': 'Acura RL Sedan 2012',
   'make': 'Acura',
   'model': 'RL Sedan',
   'make_model': 'Acura RL Sedan',
   'year': 2012}),
 (2,
  {'class_name': 'Acura TL Sedan 2012',
   'make': 'Acura',
   'model': 'TL Sedan',
   'make_model': 'Acura TL Sedan',
   'year': 2012})]

# CompCars

## Decision: CompCars Used for Exploration Only

During data preparation, we considered using CompCars as an additional training dataset.  
However, we decided not to merge it with Stanford Cars in the first baseline version because of label consistency issues.

CompCars provides make and model names, but its naming conventions differ significantly from Stanford Cars.

Examples:

- `Benz C Class` vs `Mercedes-Benz C-Class Sedan 2012`
- `BMW 3 Series` vs `BMW 3 Series Sedan 2012`
- `BMW 3 Series Convertible` could be incorrectly matched to `BMW 1 Series Convertible`

In an initial fuzzy string matching experiment, the matching recall was only around 32%. More importantly, some of the matched labels were incorrect.

For a supervised classification problem, incorrect labels are very dangerous. The model treats every label as ground truth, so noisy labels would teach the model the wrong visual patterns.

Because of this, we decided to keep the first baseline clean:

- Stanford Cars is used for training, validation, and testing.
- The official Stanford test split is kept untouched.
- CompCars is used only for domain-gap exploration.
- Real-world variation from CompCars is addressed through data augmentation instead of direct dataset merging.

This decision reduces label noise and makes the evaluation more reliable.